# Tutorial 13: Observe — Conservation & Monitoring

**Tier 4 — System Dynamics**

Before you can steer a system, you must **observe** it.
This tutorial introduces the two complementary observation layers:

| Layer | Module | Question |
|-------|--------|----------|
| **Noether Steering Law** | `noether.py` | *Is information conserved?* |
| **System Monitor** | `hllset_dynamics.py` | *How is the system changing?* |

Together they answer: **"Is my system healthy?"**

## What You'll Learn

1. **NoetherEvolution** — Track $R(t+1) = [R(t) \setminus D(t)] \cup N(t)$ step by step
2. **FluxRecord & SteeringDiagnostics** — Measure information flux $\Phi(t) = |N(t)| - |D(t)|$
3. **SteeringPhase** — Classify system phase: Balanced, Growth, Decay, Volatile
4. **Conservation Checking** — Verify the Noether invariant
5. **SystemMonitor** — Combine D/R/N with Noether into unified observations
6. **Anomaly Detection** — Automatic alerts for high flux and low retention

## Prerequisites

| Tutorial | Concept |
|----------|---------|
| [01_hllset.ipynb](01_hllset.ipynb) | HLLSet basics |
| [07_bss.ipynb](07_bss.ipynb) | Bell State Similarity (τ, ρ) |
| [09_hllset_debruijn.ipynb](09_hllset_debruijn.ipynb) | D/R/N triples |
| [11_hllset_debruijn.ipynb](11_hllset_debruijn.ipynb) | Evolution phase classification |

## Architecture

```
         additions N(t)     deletions D(t)
              │                   │
              ▼                   ▼
    ┌─────────────────────────────────────┐
    │        NoetherEvolution.step()      │
    │  R(t+1) = [R(t) \ D(t)] ∪ N(t)      │
    │                                     │
    │  → FluxRecord(Φ, |R|, popcount)     │
    │  → SteeringDiagnostics(phase, drift)│
    └──────────────┬──────────────────────┘
                   │
                   ▼
    ┌──────────────────────────────────────┐
    │         SystemMonitor.observe()      │
    │  + D/R/N decomposition               │
    │  + BSS(R(t), R(t+1))                 │
    │  + EvolutionPhase classification     │
    │  → SystemObservation (unified)       │
    │  → Anomaly detection                 │
    └──────────────────────────────────────┘
```

In [1]:
# Setup
import sys
sys.path.insert(0, '..')

from core.noether import (
    NoetherEvolution,
    SteeringPhase,
    FluxRecord,
    SteeringDiagnostics,
    compute_flux,
    apply_transition,
    is_balanced,
)
from core.hllset_dynamics import (
    SystemObservation,
    SystemMonitor,
)
from core.hllset import HLLSet
from core.bss import bss

P_BITS = 10

## 1. NoetherEvolution — First Steps

The `NoetherEvolution` engine tracks state evolution and conservation.
Create one with empty state and evolve it:

In [2]:
# Create an empty evolution engine
evo = NoetherEvolution(p_bits=P_BITS)
print(f"Initial state: {evo}")
print(f"Step count: {evo.step_count}")
print(f"Cumulative flux: {evo.cumulative_flux}")

# First step: add some tokens
additions = HLLSet.from_batch(["alpha", "beta", "gamma", "delta"], p_bits=P_BITS)
diag = evo.step(additions=additions)

print(f"\nAfter step 1:")
print(f"  Flux Φ(1) = {diag.flux:+.1f}  (added ≈{diag.flux:.0f} tokens)")
print(f"  State |R| ≈ {diag.state_card:.1f}")
print(f"  Phase: {diag.phase.value}")
print(f"  Popcount: {diag.popcount}")
print(f"  State ID: {diag.state_id[:16]}...")

Initial state: NoetherEvolution(steps=0, ΣΦ=+0.0, |R|≈0.0)
Step count: 0
Cumulative flux: 0.0

After step 1:
  Flux Φ(1) = +5.0  (added ≈5 tokens)
  State |R| ≈ 5.0
  Phase: balanced
  Popcount: 4
  State ID: 9115859c5e68dfff...


In [3]:
# Step 2: balanced update (add and delete roughly equal)
add2 = HLLSet.from_batch(["epsilon", "zeta"], p_bits=P_BITS)
del2 = HLLSet.from_batch(["alpha", "beta"], p_bits=P_BITS)

diag2 = evo.step(additions=add2, deletions=del2)
print(f"Step 2: Φ = {diag2.flux:+.1f}, ΣΦ = {diag2.cumulative_flux:+.1f}, "
      f"phase = {diag2.phase.value}")

# Step 3: pure deletion (decay)
del3 = HLLSet.from_batch(["gamma"], p_bits=P_BITS)
diag3 = evo.step(deletions=del3)
print(f"Step 3: Φ = {diag3.flux:+.1f}, ΣΦ = {diag3.cumulative_flux:+.1f}, "
      f"phase = {diag3.phase.value}")

# Step 4: growth burst
add4 = HLLSet.from_batch(["eta", "theta", "iota", "kappa", "lambda"], p_bits=P_BITS)
diag4 = evo.step(additions=add4)
print(f"Step 4: Φ = {diag4.flux:+.1f}, ΣΦ = {diag4.cumulative_flux:+.1f}, "
      f"phase = {diag4.phase.value}")

print(f"\nEngine state: {evo}")

Step 2: Φ = +1.0, ΣΦ = +6.0, phase = balanced
Step 3: Φ = -2.0, ΣΦ = +4.0, phase = growth
Step 4: Φ = +5.0, ΣΦ = +9.0, phase = growth

Engine state: NoetherEvolution(steps=4, ΣΦ=+9.0, |R|≈8.0)


## 2. step_with_tokens — Convenience API

Instead of building HLLSets manually, pass raw token lists:

In [4]:
# Fresh engine for clean demo
evo2 = NoetherEvolution(p_bits=P_BITS)

# Three steps using token lists directly
d1 = evo2.step_with_tokens(add_tokens=["cat", "dog", "bird"])
d2 = evo2.step_with_tokens(add_tokens=["fish", "snake"], del_tokens=["cat"])
d3 = evo2.step_with_tokens(del_tokens=["dog", "bird"])

for d in [d1, d2, d3]:
    print(f"Step {d.step}: Φ={d.flux:+.1f}, ΣΦ={d.cumulative_flux:+.1f}, "
          f"|R|≈{d.state_card:.1f}, phase={d.phase.value}")

Step 1: Φ=+5.0, ΣΦ=+5.0, |R|≈5.0, phase=balanced
Step 2: Φ=+1.0, ΣΦ=+6.0, |R|≈5.0, phase=balanced
Step 3: Φ=-3.0, ΣΦ=+3.0, |R|≈3.0, phase=volatile


## 3. FluxRecord — The History Tape

Every call to `step()` appends a `FluxRecord` to the engine's history.
Each record captures the full state at that time step:

In [5]:
# Inspect the history tape
print(f"History length: {len(evo2.history)}")
print(f"\n{'Step':>4}  {'Φ':>6}  {'|N|':>6}  {'|D|':>6}  {'|R|':>8}  {'Pop':>6}  State ID")
print("-" * 70)

for i, rec in enumerate(evo2.history, 1):
    print(f"{i:4d}  {rec.flux:+6.1f}  {rec.added_card:6.1f}  {rec.deleted_card:6.1f}  "
          f"{rec.state_card:8.1f}  {rec.popcount:6d}  {rec.state_id[:12]}...")

History length: 3

Step       Φ     |N|     |D|       |R|     Pop  State ID
----------------------------------------------------------------------
   1    +5.0     5.0     0.0       5.0       3  cc95abb830d0...
   2    +1.0     3.0     2.0       5.0       4  b5efde3602a0...
   3    -3.0     0.0     3.0       3.0       2  09e5d24546af...


## 4. SteeringPhase — Reading the System Pulse

The engine classifies the system into one of four phases:

| Phase | Meaning | Drift Rate |
|-------|---------|------------|
| `BALANCED` | $\Phi \approx 0$, stable evolution | $\|\bar\Phi\| < 0.5$ |
| `GROWTH` | $\Phi > 0$ sustained, system expanding | $\bar\Phi > 1.0$ |
| `DECAY` | $\Phi < 0$ sustained, system contracting | $\bar\Phi < -1.0$ |
| `VOLATILE` | $\Phi$ oscillating, unstable | high variance |

Let's drive the system through different phases:

In [6]:
# Build a longer trajectory to trigger phase detection
evo3 = NoetherEvolution(p_bits=P_BITS)

# Phase 1: Growth burst (add many tokens)
phases_seen = []
for i in range(10):
    tokens = [f"grow_{i}_{j}" for j in range(5)]
    d = evo3.step_with_tokens(add_tokens=tokens)
    phases_seen.append(d.phase.value)

# Phase 2: Balanced (equal add/delete)
for i in range(10):
    add = [f"bal_new_{i}_{j}" for j in range(3)]
    rm = [f"grow_{i}_{j}" for j in range(3)]
    d = evo3.step_with_tokens(add_tokens=add, del_tokens=rm)
    phases_seen.append(d.phase.value)

# Phase 3: Decay (only delete)
for i in range(10):
    rm = [f"bal_new_{i}_{j}" for j in range(3)]
    d = evo3.step_with_tokens(del_tokens=rm)
    phases_seen.append(d.phase.value)

# Show phase timeline
print("Phase timeline (30 steps):")
for i, ph in enumerate(phases_seen, 1):
    marker = "📈" if ph == "growth" else "⚖️" if ph == "balanced" else "📉" if ph == "decay" else "⚡"
    print(f"  Step {i:2d}: {marker} {ph}")

Phase timeline (30 steps):
  Step  1: ⚖️ balanced
  Step  2: ⚖️ balanced
  Step  3: 📈 growth
  Step  4: 📈 growth
  Step  5: 📈 growth
  Step  6: 📈 growth
  Step  7: 📈 growth
  Step  8: 📈 growth
  Step  9: 📈 growth
  Step 10: 📈 growth
  Step 11: 📈 growth
  Step 12: 📈 growth
  Step 13: 📈 growth
  Step 14: 📈 growth
  Step 15: 📈 growth
  Step 16: 📈 growth
  Step 17: 📈 growth
  Step 18: 📈 growth
  Step 19: 📈 growth
  Step 20: 📈 growth
  Step 21: 📈 growth
  Step 22: 📈 growth
  Step 23: 📈 growth
  Step 24: 📈 growth
  Step 25: 📈 growth
  Step 26: 📈 growth
  Step 27: 📈 growth
  Step 28: ⚡ volatile
  Step 29: ⚡ volatile
  Step 30: ⚡ volatile


## 5. SteeringDiagnostics — Decision Support

The `SteeringDiagnostics` object returned by `step()` gives applications
everything they need to decide whether to intervene:

In [7]:
# Get the latest diagnostics
d = evo3.step_with_tokens(add_tokens=["test_token"])

print("=== SteeringDiagnostics ===")
print(f"  step:               {d.step}")
print(f"  flux Φ(t):          {d.flux:+.1f}")
print(f"  cumulative ΣΦ:      {d.cumulative_flux:+.1f}")
print(f"  state |R|:          {d.state_card:.1f}")
print(f"  popcount:           {d.popcount}")
print(f"  phase:              {d.phase.value}")
print(f"  drift_rate:         {d.drift_rate:.3f}")
print(f"  conservation_error: {d.conservation_error:.3f}")
print(f"  is_balanced:        {d.is_balanced()}")
print(f"  state_id:           {d.state_id[:20]}...")

=== SteeringDiagnostics ===
  step:               31
  flux Φ(t):          +2.0
  cumulative ΣΦ:      +21.0
  state |R|:          20.0
  popcount:           20
  phase:              volatile
  drift_rate:         0.677
  conservation_error: 1.000
  is_balanced:        True
  state_id:           852542057044affb90ec...


## 6. conservation_check — The Noether Invariant

**Theorem 4.1 (Noether Steering Law):**
If $|N(t)| = |D(t)|$ for all $t$, then $\sum_i R_i(t)$ is conserved
(modulo hash collisions).

The `conservation_check()` method verifies this:

In [8]:
# Fresh engine with perfectly balanced updates
evo_bal = NoetherEvolution(p_bits=P_BITS)

# Seed the system
evo_bal.step_with_tokens(add_tokens=[f"token_{i}" for i in range(20)])

# Apply 10 balanced updates (swap equal-sized batches)
for i in range(10):
    add = [f"new_{i}_{j}" for j in range(3)]
    rm = [f"token_{i*2 + j}" for j in range(3)]
    evo_bal.step_with_tokens(add_tokens=add, del_tokens=rm)

# Check conservation
check = evo_bal.conservation_check()
print("=== Conservation Check ===")
for k, v in check.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.4f}")
    else:
        print(f"  {k:25s}: {v}")

=== Conservation Check ===
  popcount_initial         : 0
  popcount_current         : 29
  cumulative_flux          : 16.0000
  expected_popcount        : 16.0000
  conservation_error       : 13.0000
  is_conserved             : False
  density                  : 0.0008
  collision_estimate       : 0.0206
  steps                    : 11


## 7. flux_statistics — Aggregate Flux Analysis

For longer runs, `flux_statistics()` gives a statistical summary
including mean, standard deviation, and trend:

In [9]:
# Use the phase engine (evo3) which has 31 steps of varied behavior
stats = evo3.flux_statistics()
print("=== Flux Statistics ===")
for k, v in stats.items():
    if isinstance(v, float):
        print(f"  {k:10s}: {v:+.3f}")
    else:
        print(f"  {k:10s}: {v}")

print(f"\nInterpretation:")
print(f"  Mean flux {stats['mean']:+.2f} → {'growing' if stats['mean'] > 0.5 else 'shrinking' if stats['mean'] < -0.5 else 'roughly balanced'}")
print(f"  Trend {stats['trend']:+.4f} → {'accelerating' if stats['trend'] > 0.01 else 'decelerating' if stats['trend'] < -0.01 else 'stable'}")

=== Flux Statistics ===
  mean      : +0.677
  std       : +4.011
  min       : -5.000
  max       : +7.000
  trend     : -0.386
  count     : 31

Interpretation:
  Mean flux +0.68 → growing
  Trend -0.3859 → decelerating


## 8. Standalone Convenience Functions

Three functions work without creating a `NoetherEvolution` engine:

In [10]:
# compute_flux — measure Φ without stepping
add = HLLSet.from_batch(["x", "y", "z"], p_bits=P_BITS)
rm  = HLLSet.from_batch(["a"], p_bits=P_BITS)
phi = compute_flux(add, rm)
print(f"compute_flux: Φ = {phi:+.1f}")

# is_balanced — quick check
print(f"is_balanced:  {is_balanced(add, rm)}")
print(f"is_balanced (equal): {is_balanced(add, add)}")

# apply_transition — pure state transform
state = HLLSet.from_batch(["a", "b", "c"], p_bits=P_BITS)
new_state = apply_transition(state, additions=add, deletions=rm)
print(f"\napply_transition:")
print(f"  Before: |R| ≈ {state.cardinality():.1f}")
print(f"  After:  |R| ≈ {new_state.cardinality():.1f}")

compute_flux: Φ = +2.0
is_balanced:  False
is_balanced (equal): True

apply_transition:
  Before: |R| ≈ 4.0
  After:  |R| ≈ 6.0


## 9. snapshot & reset — Lifecycle Management

`snapshot()` captures the engine state for serialization.
`reset()` clears history and starts fresh:

In [11]:
# Snapshot the balanced evolution engine
snap = evo_bal.snapshot()
print(f"Snapshot keys: {list(snap.keys())}")
print(f"  p_bits:          {snap['p_bits']}")
print(f"  step_count:      {snap['step_count']}")
print(f"  cumulative_flux: {snap['cumulative_flux']:+.1f}")
print(f"  history records: {len(snap['history'])}")
print(f"  state bytes:     {len(snap['state_registers'])}")

# Reset
evo_bal.reset()
print(f"\nAfter reset:")
print(f"  step_count:      {evo_bal.step_count}")
print(f"  cumulative_flux: {evo_bal.cumulative_flux}")
print(f"  history length:  {len(evo_bal.history)}")

Snapshot keys: ['state_registers', 'p_bits', 'step_count', 'cumulative_flux', 'initial_popcount', 'history']
  p_bits:          10
  step_count:      11
  cumulative_flux: +16.0
  history records: 11
  state bytes:     4096

After reset:
  step_count:      0
  cumulative_flux: 0.0
  history length:  0


---

## 10. SystemMonitor — Unified Observation Layer

The `SystemMonitor` from `hllset_dynamics.py` wraps `NoetherEvolution`
with D/R/N decomposition, BSS tracking, and anomaly detection:

```
SystemMonitor.observe(state, time)
    ├── decompose_transition(prev, state) → DRNTriple
    ├── bss(prev, state) → BSSPair
    ├── classify_transition(drn) → EvolutionPhase
    ├── noether.step(additions, deletions) → flux, ΣΦ
    └── → SystemObservation (all-in-one)
```

In [12]:
# Create a SystemMonitor
monitor = SystemMonitor(p_bits=P_BITS)

# Simulate a document evolving over time
versions = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "cat", "sat", "on", "a", "rug"],           # "the mat" → "a rug"
    ["the", "dog", "sat", "on", "a", "rug"],            # "cat" → "dog"
    ["a", "dog", "ran", "across", "the", "field"],      # major rewrite
    ["a", "dog", "ran", "across", "the", "field", "quickly"],  # small addition
]

print(f"{'Time':>4}  {'|R|':>6}  {'Phase':>12}  {'τ':>5}  {'Φ':>6}  {'ΣΦ':>6}  {'Cost':>6}  {'Retain':>6}")
print("-" * 65)

for t, tokens in enumerate(versions):
    state = HLLSet.from_batch(tokens, p_bits=P_BITS)
    obs = monitor.observe(state, timestamp=float(t))
    
    tau = f"{obs.bss_from_prev.tau:.2f}" if obs.bss_from_prev else "  —"
    phase = obs.phase.value if obs.phase else "initial"
    
    print(f"{t:4d}  {obs.cardinality:6.1f}  {phase:>12}  {tau:>5}  "
          f"{obs.flux:+6.1f}  {obs.cumulative_flux:+6.1f}  "
          f"{obs.transition_cost:6.1f}  {obs.retention_ratio:6.2f}")

Time     |R|         Phase      τ       Φ      ΣΦ    Cost  Retain
-----------------------------------------------------------------
   0     6.0       initial      —    +6.0    +6.0     0.0    1.00
   1     6.0        stable   0.83    +0.0    +6.0     4.0    0.71
   2     6.0        stable   0.83    +0.0    +6.0     4.0    0.71
   3     7.0        stable   0.57    +1.0    +7.0     7.0    0.57
   4     7.0        growth   1.00    +2.0    +9.0     2.0    1.00


## 11. SystemObservation — Rich State Snapshots

Each `SystemObservation` carries everything you need for decision-making:

In [13]:
# Inspect the last observation (the "quickly" addition)
obs = monitor.history[-1]

print("=== SystemObservation ===")
print(f"  timestamp:       {obs.timestamp}")
print(f"  cardinality:     {obs.cardinality:.1f}")
print(f"  is_stable:       {obs.is_stable}")
print(f"  transition_cost: {obs.transition_cost:.1f}")
print(f"  retention_ratio: {obs.retention_ratio:.2f}")
print(f"  flux:            {obs.flux:+.1f}")
print(f"  cumulative_flux: {obs.cumulative_flux:+.1f}")

if obs.drn is not None:
    print(f"\n  D/R/N breakdown:")
    print(f"    Deleted: ≈{obs.drn.deleted_card:.1f}")
    print(f"    Retained: ≈{obs.drn.retained_card:.1f}")
    print(f"    Novel:    ≈{obs.drn.novel_card:.1f}")

if obs.bss_from_prev is not None:
    print(f"\n  BSS from previous:")
    print(f"    τ = {obs.bss_from_prev.tau:.4f}")
    print(f"    ρ = {obs.bss_from_prev.rho:.4f}")

if obs.phase is not None:
    print(f"\n  Evolution phase: {obs.phase.value}")

=== SystemObservation ===
  timestamp:       4.0
  cardinality:     7.0
  is_stable:       False
  transition_cost: 2.0
  retention_ratio: 1.00
  flux:            +2.0
  cumulative_flux: +9.0

  D/R/N breakdown:
    Deleted: ≈0.0
    Retained: ≈7.0
    Novel:    ≈2.0

  BSS from previous:
    τ = 1.0000
    ρ = 0.0000

  Evolution phase: growth


## 12. Anomaly Detection

The monitor raises alerts when metrics exceed thresholds:

## 13. Monitor Summary — System Health Report

Call `summary()` for aggregate statistics:

In [14]:
# Use the document monitor from section 10
summary = monitor.summary()
print("=== System Health Report ===")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.3f}")
    elif isinstance(v, dict):
        print(f"  {k:25s}:")
        for pk, pv in v.items():
            print(f"    {pk:20s}: {pv}")
    else:
        print(f"  {k:25s}: {v}")

=== System Health Report ===
  observations             : 5
  total_flux               : 9.000
  avg_transition_cost      : 3.400
  avg_retention            : 0.800
  phase_distribution       :
    stable              : 3
    growth              : 1
  final_cardinality        : 7


## 14. End-to-End: Monitoring a Streaming Corpus

Let's simulate a real-time document stream and monitor its health:

In [15]:
import random
random.seed(42)

# Simulate 20-step stream: documents arrive, old ones expire
stream_monitor = SystemMonitor(p_bits=P_BITS)
vocab = [f"word_{i}" for i in range(100)]
current_tokens = set(random.sample(vocab, 15))

all_anomalies = []

for t in range(20):
    # Random evolution: add 2-5 tokens, remove 0-4 tokens
    n_add = random.randint(2, 5)
    n_del = random.randint(0, 4)
    
    to_add = set(random.sample(vocab, n_add))
    to_del = set(random.sample(list(current_tokens), min(n_del, len(current_tokens))))
    
    current_tokens = (current_tokens - to_del) | to_add
    state = HLLSet.from_batch(list(current_tokens), p_bits=P_BITS)
    
    obs = stream_monitor.observe(state, timestamp=float(t))
    anomalies = stream_monitor.detect_anomalies(obs)
    if anomalies:
        all_anomalies.append((t, anomalies))

# Results
print(f"Stream: 20 time steps")
print(f"Anomalies detected: {len(all_anomalies)}")
for t, anoms in all_anomalies:
    for a in anoms:
        print(f"  t={t}: {a}")

# Final health report
print(f"\n=== Final Health Report ===")
sm = stream_monitor.summary()
print(f"  Observations:       {sm['observations']}")
print(f"  Total flux:         {sm['total_flux']:+.1f}")
print(f"  Avg transition cost:{sm['avg_transition_cost']:.1f}")
print(f"  Avg retention:      {sm['avg_retention']:.3f}")
print(f"  Final cardinality:  {sm['final_cardinality']:.1f}")
if sm.get('phase_distribution'):
    print(f"  Phase distribution:")
    for phase, count in sm['phase_distribution'].items():
        bar = "█" * count
        print(f"    {phase:12s}: {count:2d} {bar}")

Stream: 20 time steps
Anomalies detected: 0

=== Final Health Report ===
  Observations:       20
  Total flux:         +38.0
  Avg transition cost:6.3
  Avg retention:      0.905
  Final cardinality:  33.0
  Phase distribution:
    stable      : 14 ██████████████
    growth      :  5 █████


## 15. Noether + Monitor: Combined View

The `SystemMonitor` embeds a `NoetherEvolution` engine internally.
You can access the Noether layer for deeper conservation analysis:

In [16]:
# Access the embedded Noether engine
noether = stream_monitor.noether

print("=== Noether Conservation (via Monitor) ===")
check = noether.conservation_check()
print(f"  Popcount initial:   {check['popcount_initial']}")
print(f"  Popcount current:   {check['popcount_current']}")
print(f"  Cumulative flux:    {check['cumulative_flux']:+.1f}")
print(f"  Expected popcount:  {check['expected_popcount']:.1f}")
print(f"  Conservation error: {check['conservation_error']:.3f}")
print(f"  Is conserved:       {check['is_conserved']}")

print("\n=== Noether Flux Statistics ===")
fstats = noether.flux_statistics()
print(f"  Mean Φ:    {fstats['mean']:+.3f}")
print(f"  Std Φ:     {fstats['std']:.3f}")
print(f"  Range:     [{fstats['min']:+.1f}, {fstats['max']:+.1f}]")
print(f"  Trend:     {fstats['trend']:+.4f}")

=== Noether Conservation (via Monitor) ===
  Popcount initial:   0
  Popcount current:   31
  Cumulative flux:    +38.0
  Expected popcount:  38.0
  Conservation error: 7.000
  Is conserved:       False

=== Noether Flux Statistics ===
  Mean Φ:    +1.900
  Std Φ:     4.288
  Range:     [-3.0, +17.0]
  Trend:     -0.3143


---

## Summary & Key Takeaways

### Noether Layer (Conservation)

| API | Purpose |
|-----|---------|
| `NoetherEvolution(p_bits)` | Create evolution engine |
| `.step(additions, deletions)` | Evolve state, get diagnostics |
| `.step_with_tokens(add, del)` | Convenience: token lists |
| `.conservation_check()` | Verify Noether invariant |
| `.flux_statistics()` | Statistical flux summary |
| `.history` | Full `FluxRecord` tape |
| `.snapshot()` / `.reset()` | Lifecycle management |
| `compute_flux(N, D)` | Standalone Φ computation |
| `apply_transition(R, N, D)` | Pure state transform |
| `is_balanced(N, D)` | Quick balance check |

### Monitor Layer (Observation)

| API | Purpose |
|-----|---------|
| `SystemMonitor(p_bits)` | Create unified monitor |
| `.observe(state, time)` | Record observation with D/R/N + BSS + Noether |
| `.detect_anomalies(obs)` | Check for HIGH_FLUX, LOW_RETENTION, REPLACEMENT |
| `.summary()` | Aggregate health report |
| `.noether` | Access embedded Noether engine |

### The Noether Steering Law

$$\text{If } |N(t)| = |D(t)| \text{ for all } t, \text{ then } \sum_i R_i(t) \text{ is conserved.}$$

Deviation from balance ($|\Phi| > 0$) signals:
- **Growth** → system is accumulating information
- **Decay** → system is losing information  
- **Volatile** → system is unstable

**Next:** [14_act.ipynb](14_act.ipynb) — Planning, Steering & Evolution

## 12. Anomaly Detection

The monitor raises alerts when metrics exceed thresholds: